## Anwendungsbeispiel Vektordatenbanken - Milvus

### Starten einer Standalone Milvus Instanz in einem Docker Container
Nutzen Sie die `docker-compose.yml` Datei aus dem Repository und starten Sie einen Container per `docker compose up -d`. 


Per Python lässt sich die Verbindung folgendermaßen testen: 

In [1]:
from pymilvus import MilvusClient

client = MilvusClient(uri="http://localhost:19530")
print("connected")

connected


### Wikipedia Datensatz laden 
Für dieses Tutorial verwenden wir einen öffentlichen Wikipedia-Datensatz, der über die HuggingFace-Datasets-Bibliothek verfügbar ist.
Der Datensatz enthält Artikel aus der Simple English Wikipedia und eignet sich gut für Demonstrationen von semantischer Suche.

Die Parameter bedeuten:
- "wikipedia" – Name des Datensatzes
- "20220301.simple" – Version und Sprachvariante (Simple English Wikipedia vom März 2022)
- split="train" – Auswahl des Datensatzteils (hier der Hauptdatensatz)

In [21]:
from datasets import load_dataset

ds = load_dataset(
    "wikipedia", "20220301.simple" ,split="train"
)

ds

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 205328
})


### Verbindung zur Datenbank
Nachdem der Docker-Container namens milvus an Port 19530 gestartet ist, kann über die Verbindung eine neue Datenbank erstellt werden. Wir nennen diese "wiki_demo". Falls die Datenbank schon erstellt wurde, können wir sie per `using_database("wiki_demo")` direkt referenzieren.

In [32]:
from pymilvus import DataType, utility

client.create_database("wiki_demo")
#client.using_database("wiki_demo")

### Schema erstellen

Im folgenden wird das Schema einer Milvus-Collection definiert. Dieses beschreibt die Struktur der Daten, die später in der Datenbank gespeichert werden. 

In [33]:
schema = MilvusClient.create_schema(
    # automatische Generierung der ID-Spalte
    auto_id=True,
)

Insgesamt werden fünf Felder definiert. 

In [34]:
# id
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
# url
schema.add_field(field_name="url", datatype=DataType.VARCHAR,  max_length=500, description="article URL")
# title
schema.add_field(field_name="title", datatype=DataType.VARCHAR, max_length=200, description="article title")
# text
schema.add_field(field_name="text", datatype=DataType.VARCHAR,  max_length=50000, description="article text")
# embedding
schema.add_field(field_name="embedding", datatype=DataType.FLOAT_VECTOR, dim=384, description="embedding vector")


{'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'url', 'description': 'article URL', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 500}}, {'name': 'title', 'description': 'article title', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 200}}, {'name': 'text', 'description': 'article text', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 50000}}, {'name': 'embedding', 'description': 'embedding vector', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 384}}], 'enable_dynamic_field': False}

### Definieren des Index
`field_name="embedding"` : Dies gibt das Feld in der Datenbank- oder dem Collection-Schema an, das Vektoren enthält. In der Regel handelt es sich dabei um hochdimensionale Vektoren, die z. B. von einem Embedding-Modell (wie z. B. BERT oder OpenAI-Embeddings) erzeugt wurden. Dieses Feld soll dann indexiert werden, damit man später effizient darauf suchen kann.

`index_type="AUTOINDEX"`
Mit diesem Index-Typ übernimmt das Datenbanksystem die Auswahl des besten Indexverfahrens. Das System analysiert automatisch die Daten und entscheidet, welche Indexstruktur (z. B. IVF, HNSW, etc.) für die Anforderungen am besten geeignet ist. 

`metric_type="IP"`
„IP“ steht für Inner Product (Skalarprodukt). Dies ist eine Methode zur Berechnung der Ähnlichkeit zwischen zwei Vektoren. Besonders häufig wird sie verwendet, wenn die Vektoren normalisiert sind (d. h. ihre Länge = 1). In diesem Fall entspricht das Skalarprodukt der Kosinusähnlichkeit (Cosine Similarity), was sehr nützlich für semantische Suchen ist (z. B. bei Text-Embeddings).

In [35]:
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="embedding",
    index_type="AUTOINDEX",
    metric_type="IP",
)

### Sammlung erstellen

In diesem Schritt wird eine Collection in Milvus erstellt. Eine Collection ist vergleichbar mit einer Tabelle in einer Datenbank und speichert die Artikel und deren Embeddings.

In [36]:
collection_name = "test_wiki"

if client.has_collection(collection_name):
    client.drop_collection(collection_name)


embedding_dim = 384
collection = client.create_collection(
    collection_name=collection_name,
    #dimension=embedding_dim,
    schema=schema,
    index_params=index_params,
)

In [14]:
collection_desc = client.describe_collection(
    collection_name=collection_name
)
print(collection_desc)

{'collection_name': 'test_wiki', 'auto_id': True, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 100, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 101, 'name': 'url', 'description': 'article URL', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 500}}, {'field_id': 102, 'name': 'title', 'description': 'article title', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 200}}, {'field_id': 103, 'name': 'text', 'description': 'article text', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 50000}}, {'field_id': 104, 'name': 'embedding', 'description': 'embedding vector', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 384}}], 'functions': [], 'aliases': [], 'collection_id': 464816993002067232, 'consistency_level': 2, 'properties': {}, 'num_partitions': 1, 'enable_dynamic_field': False}


### Text in Vektoren umwandeln

In diesem Abschnitt wird ein Embedding-Modell geladen, nämlich `all-MiniLM-L6-v2`. Dieses wandelt Text in numerische Vektoren um, die in Milvus gespeichert werden und eine semantische Suche ermöglichen. 

Das Modell erzeugt 384-dimensionale Embeddings, die relativ klein und schnell zu verarbeiten sind. Es ist gut geeignet für die semantische Textsuche.

Das Modell wird unter dem Namen transformer gespeichert, damit es im nächsten Schritt verwendet werden kann

In [37]:
from sentence_transformers import SentenceTransformer
transformer = SentenceTransformer('all-MiniLM-L6-v2')

def emb_text(text): 
    return transformer.encode(text)

### Embeddings berechnen und Daten in Milvus speichern 

In diesem Schritt werden Texte aus dem Datensatz in Embeddings umgewandelt und anschließend in die Milvus-Collection eingefügt.

In [38]:
def embed_insert(ds, limit=500):
    """
    Berechnet Embeddings für Dokumente aus einem Datensatz und fügt sie
    zusammen mit Metadaten in eine Milvus-Collection ein.
    
    Für die Demo wird nur eine Teilmenge des Datensatzes verwendet.
    """
    
    rows = []
    
    # Reduzierung der Anzahl der Dokumente für eine Demo
    ds1 = ds.select(range(limit))
    
    # Für jedes Dokument ein Embedding berechnen und Datensatz vorbereiten
    for item in ds1:
        rows.append({
            "url": item['url'],
            "title": item['title'],
            "text": item['text'],
            "embedding": emb_text(item['text'])
        })
    
    # Datensätze in die Milvus-Collection einfügen
    client.insert(collection_name, rows)
    
    # Sicherstellen, dass die Daten gespeichert werden
    client.flush(collection_name)


### Alternative: Batch-Verarbeitung

Die Funktion wählt einen Teil des Datensatzes aus, berechnet für jeden Text ein Embedding und speichert anschließend alle Datensätze gesammelt in der Milvus-Collection.

In [39]:
def embed_insert_batch(ds, limit=500):
    """
    Wandelt Texte in Embeddings um und speichert sie in der Milvus-Collection.
    Für das Tutorial wird nur ein Teil des Datensatzes verwendet, damit der
    Code schneller ausgeführt werden kann.
    """

    # Teilmenge des Datensatzes auswählen
    ds_subset = ds.select(range(limit))

    # Texte extrahieren und Embeddings im Batch berechnen
    texts = [item["text"] for item in ds_subset]
    embeddings = transformer.encode(texts)

    # Datensätze für das Einfügen vorbereiten
    rows = [
        {
            "url": item["url"],
            "title": item["title"],
            "text": item["text"],
            "embedding": emb
        }
        for item, emb in zip(ds_subset, embeddings)
    ]

    # Daten in die Milvus-Collection einfügen
    client.insert(collection_name, rows)

    # Sicherstellen, dass die Daten dauerhaft gespeichert werden
    client.flush(collection_name)

Aufruf der Funktionen (je nachdem ob einzelne Verarbeitung oder per Batch)

In [40]:
# Aufruf der Funktion zum Einfügen der Daten
#embed_insert(ds)
embed_insert_batch(ds, limit=1000)

Mit `describe_collection()` können Informationen über die Struktur der Collection abgefragt werden, z. B. das Schema und die definierten Felder.

In [17]:
res = client.describe_collection(
    collection_name=collection_name
)

print(res)

{'collection_name': 'test_wiki', 'auto_id': True, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 100, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 101, 'name': 'url', 'description': 'article URL', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 500}}, {'field_id': 102, 'name': 'title', 'description': 'article title', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 200}}, {'field_id': 103, 'name': 'text', 'description': 'article text', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 50000}}, {'field_id': 104, 'name': 'embedding', 'description': 'embedding vector', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 384}}], 'functions': [], 'aliases': [], 'collection_id': 464816993002067232, 'consistency_level': 2, 'properties': {}, 'num_partitions': 1, 'enable_dynamic_field': False}


### Suche
Die Collection kann jetzt durchsucht werden. Dazu muss die Anfrage ebenfalls in ein Embedding umgewandelt werden. 

In [41]:
# Beispiel für eine Anfrage
question = "What does air consist of?"

Für den Start einer Ähnlichkeitssuche muss noch die Collection in den Hauptspeicher geladen werden. 

In [42]:
client.load_collection(collection_name)
client.get_collection_stats(collection_name)

{'row_count': 1000}

In diesem Schritt wird eine Vektorsuche in der Milvus-Collection durchgeführt. Dabei wird eine natürliche Frage in ein Embedding umgewandelt und mit den gespeicherten Dokumenten verglichen.

In [43]:

search_res = client.search(
    collection_name=collection_name,
    data=[
        emb_text(question)
    ],  # Umwandlung der Frage in einen Embedding-Vektor
    limit=4,  # Anzahl der zurückgegebenen ähnlichsten Treffer
    search_params={"metric_type": "IP", "params": {}},  # Ähnlichkeitsmaß: Inner Product
    output_fields=["text"],  # Welche Felder aus der Collection zurückgegeben werden sollen
)

Die äußere Schleife durchläuft die einzelnen Suchanfragen, die innere Schleife die gefundenen Dokumente.

In [44]:
for hits in search_res:
    for hit in hits: 
        print(hit)

{'id': 464816993002071411, 'distance': 0.6168864965438843, 'entity': {'text': "Air refers to the Earth's atmosphere. Air is a mixture of many gases and tiny dust particles. It is the clear gas in which living things live and breathe. It has an indefinite shape and volume. It has mass and weight, because it is matter. The weight of air creates atmospheric  pressure. There is no air in outer space. \n\nAir is a mixture of about 78% of nitrogen, 21% of oxygen, 0.9% of argon, 0.04% of carbon dioxide, and very small amounts of other gases.  There is an average of about 1% water vapour.\n\nAnimals live and need to breathe the oxygen in the air. In breathing, the lungs put oxygen into the blood, and send back carbon dioxide to the air. Plants need the carbon dioxide in the air to live. They give off the oxygen that we breathe. Without it we die of asphyxia.  \n\nWind is moving air, this is refreshing. This causes weather. \n\nAir can be polluted by some gases (such as carbon monoxide, hydroca

Übersichtlichere Ausgabe: Für jeden Treffer wird der Text des gefundenen Dokuments sowie die Ähnlichkeitsdistanz zur Suchanfrage angegeben. 

In [45]:
import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))



[
    [
        "Air refers to the Earth's atmosphere. Air is a mixture of many gases and tiny dust particles. It is the clear gas in which living things live and breathe. It has an indefinite shape and volume. It has mass and weight, because it is matter. The weight of air creates atmospheric  pressure. There is no air in outer space. \n\nAir is a mixture of about 78% of nitrogen, 21% of oxygen, 0.9% of argon, 0.04% of carbon dioxide, and very small amounts of other gases.  There is an average of about 1% water vapour.\n\nAnimals live and need to breathe the oxygen in the air. In breathing, the lungs put oxygen into the blood, and send back carbon dioxide to the air. Plants need the carbon dioxide in the air to live. They give off the oxygen that we breathe. Without it we die of asphyxia.  \n\nWind is moving air, this is refreshing. This causes weather. \n\nAir can be polluted by some gases (such as carbon monoxide, hydrocarbons, and nitrogen oxides), smoke, and ash. This air pollutio

### Inhalte einer Collection löschen

In [30]:
client.flush(collection_name)

### Datenbank löschen

In [31]:
client.drop_database("wiki_demo")